In [1]:
import nrrd
import numpy as np
import os
from skimage import measure
import csv
from scipy.ndimage import binary_dilation

In [2]:
def create_directory(directory_name):
    if not os.path.exists(directory_name):
        os.makedirs(directory_name)
    return None

def faces_to_edges(faces):
    edges = []
    for face in faces:
        edges.append(sorted([face[0],face[1]]))
        edges.append(sorted([face[1],face[2]]))
        edges.append(sorted([face[2],face[0]]))
    seen = set()
    unique_pairs = []
    
    for pair in edges:
        tup = tuple(pair)  # convert to tuple so it’s hashable
        if tup not in seen:
            seen.add(tup)
            unique_pairs.append(pair)
    return unique_pairs

def thicken(grid: np.ndarray) -> np.ndarray:
    structure = np.ones((3, 3, 3), dtype=bool)
    return binary_dilation(grid.astype(bool), structure=structure)

### Folder where the segmentations are

In [3]:
segmentation_folder_path = './../3Dslicer-lung-segmentations/'

### Folder where the processing will be

In [4]:
folder_path = './3Dslicer-full-lung-segmentations-processed'

In [5]:
# List all entries in the folder, then filter to keep only files
file_names = [
    fname
    for fname in os.listdir(segmentation_folder_path)
    if os.path.isfile(os.path.join(segmentation_folder_path, fname))
]

In [6]:
print(file_names)

['Lung segmentation-57.seg.nrrd', 'Lung segmentation-48.seg.nrrd', 'Lung segmentation-AI-9.seg.nrrd', 'Lung segmentation-9.seg.nrrd', 'Lung segmentation-4.seg.nrrd', 'Lung segmentation-AI-20.seg.nrrd', 'Lung segmentation-AI-29.seg.nrrd', 'Lung segmentation-AI-2.seg.nrrd', 'Lung segmentation-65.seg.nrrd', 'Lung segmentation-56.seg.nrrd', 'Lung segmentation-AI-36.seg.nrrd', 'Lung segmentation-AI-34.seg.nrrd', 'Lung segmentation-52.seg.nrrd', 'Lung segmentation-AI-19.seg.nrrd', 'Lung segmentation-AI-15.seg.nrrd', 'Lung segmentation-32.seg.nrrd', 'Lung segmentation-20.seg.nrrd', 'Lung segmentation-10.seg.nrrd', 'Lung segmentation-39.seg.nrrd', 'Lung segmentation-17.seg.nrrd', 'Lung segmentation-AI-17.seg.nrrd', 'Lung segmentation-AI-10.seg.nrrd', 'Lung segmentation-AI-4.seg.nrrd', 'Lung segmentation-AI-39.seg.nrrd', 'Lung segmentation-2.seg.nrrd', 'Lung segmentation-AI-25.seg.nrrd', 'Lung segmentation-19.seg.nrrd', 'Lung segmentation-34.seg.nrrd', 'Lung segmentation-AI-48.seg.nrrd', 'Lung 

In [8]:
for fileName in file_names:
    print(fileName)
    segmentation_file_name = fileName
    
    file_name = fileName.split('.', 1)[0]
    data, header = nrrd.read( segmentation_folder_path + segmentation_file_name )
    space_dirs = header["space directions"]
    spatial_dirs = [v for v in space_dirs if v is not None]
    spacing = [np.linalg.norm(v) for v in spatial_dirs][1:]
    masks = {}
    for i in range(data.shape[0]):
        seg_id = header.get(f"Segment{i}_ID")
        label_val = int(header.get(f"Segment{i}_LabelValue"))
        mask = data[i] == label_val
        masks[seg_id] = mask
    
    # Volume data
    lungs = masks['lungs']
    volume_data = thicken(lungs)
    
    # Isosurface value
    iso_value = 0.5

    # Extract the isosurface
    vertices, faces, normals, values = measure.marching_cubes(volume_data, 
                                                              iso_value, 
                                                              gradient_direction='ascent', 
                                                              allow_degenerate=False, 
                                                              method='lewiner',
                                                              spacing=spacing)
    
    # Create 
    edges = faces_to_edges(faces)
    create_directory(f"{folder_path}/{file_name}")
    with open(f"{folder_path}/{file_name}/vertices.csv", "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        for vertex in vertices:
            writer.writerow(vertex)
    
    with open(f"{folder_path}/{file_name}/edges.csv", "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        # Write each pair as a row in the CSV
        for edge in edges:
            writer.writerow(edge)

Lung segmentation-57.seg.nrrd
Lung segmentation-48.seg.nrrd
Lung segmentation-AI-9.seg.nrrd
Lung segmentation-9.seg.nrrd
Lung segmentation-4.seg.nrrd
Lung segmentation-AI-20.seg.nrrd
Lung segmentation-AI-29.seg.nrrd
Lung segmentation-AI-2.seg.nrrd
Lung segmentation-65.seg.nrrd
Lung segmentation-56.seg.nrrd
Lung segmentation-AI-36.seg.nrrd
Lung segmentation-AI-34.seg.nrrd
Lung segmentation-52.seg.nrrd
Lung segmentation-AI-19.seg.nrrd
Lung segmentation-AI-15.seg.nrrd
Lung segmentation-32.seg.nrrd
Lung segmentation-20.seg.nrrd
Lung segmentation-10.seg.nrrd
Lung segmentation-39.seg.nrrd
Lung segmentation-17.seg.nrrd
Lung segmentation-AI-17.seg.nrrd
Lung segmentation-AI-10.seg.nrrd
Lung segmentation-AI-4.seg.nrrd
Lung segmentation-AI-39.seg.nrrd
Lung segmentation-2.seg.nrrd
Lung segmentation-AI-25.seg.nrrd
Lung segmentation-19.seg.nrrd
Lung segmentation-34.seg.nrrd
Lung segmentation-AI-48.seg.nrrd
Lung segmentation-15.seg.nrrd
Lung segmentation-36.seg.nrrd
Lung segmentation-25.seg.nrrd
Lung